# Vision Language Models (VLM) NVIDIA NIM

Vision language models are multimodal AI systems built by combining a large language model (LLM) with a vision encoder, giving the LLM the ability to “see.” With this ability, VLMs can process and provide advanced understanding of video, image, and text inputs supplied in the prompt to generate text responses.

Most VLMs follow an architecture with three parts:

* A vision encoder
* A projector
* An LLM
The vision encoder is typically a CLIP-based model with a transformer architecture that has been trained on millions of image-text pairs, giving it the ability to associate images and text. The projector is a set of layers that translates the output of the vision encoder into a form the LLM can understand, often interpreted as image tokens. This projector can be a simple line layer like LLaVA and VILA, or something more complex like the cross-attention layers used in Llama 3.2 Vision.

Any off-the-shelf LLM can be used to build a VLM. There are hundreds of VLM variants that combine various LLMs with vision encoders.

<img src="imgs/vlm-architecture-diagram.jpg" alt="VLM Architecture" width="600"/>

## Setup Environment 

In [ ]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = os.environ["NVIDIA_API_KEY"]
    api_key = nvapi_key

In [ ]:
from time import time 
from PIL import Image

import json, io, subprocess, base64
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import requests
import pandas as pd

Ensure that no errors occured during the installation and import in the two cells above before continuing.

In [ ]:
#Setup VLM NIM Urls 
neva_api_url = "https://ai.api.nvidia.com/v1/vlm/nvidia/neva-22b"  # https://build.nvidia.com/nvidia/neva-22b
kosmos2_api_url = "https://ai.api.nvidia.com/v1/vlm/microsoft/kosmos-2" # https://build.nvidia.com/microsoft/microsoft-kosmos-2
fuyu8b_api_url = "https://ai.api.nvidia.com/v1/vlm/adept/fuyu-8b" # https://build.nvidia.com/adept/fuyu-8b
paligemma_api_url = "https://ai.api.nvidia.com/v1/vlm/google/paligemma" # https://build.nvidia.com/google/google-paligemma/modelcard
phi3_api_url = "https://ai.api.nvidia.com/v1/vlm/microsoft/phi-3-vision-128k-instruct" # https://build.nvidia.com/microsoft/phi-3-vision-128k-instruct

## Text Capabilites

This section demonstrates calling a VLM NIM with a POST request using just text input. The request will be made up of some headers that will include your API Key for authorization and then the payload with the content for the VLM. We would only use the "Language Model" component of the VLM

The API key should be presented as a Bearer token and the request body will be JSON format so we need to specify this in the header. 

In [ ]:
headers = {
  "Authorization": f"Bearer {api_key}",
  "Accept": "application/json"
}

The payload is JSON format and follows an API schema similar to the OpenAI API. For full details on the API Spec for NIMs, visit these pages:
- https://docs.api.nvidia.com/nim/reference/nvidia-neva-22b  
- https://docs.api.nvidia.com/nim/reference/microsoft-kosmos-2
- https://docs.api.nvidia.com/nim/reference/adept-fuyu-8b
- https://docs.api.nvidia.com/nim/reference/google-paligemma
- https://docs.api.nvidia.com/nim/reference/microsoft-phi-3-vision-128k-instruct

In [ ]:
payload = {
  "messages": [
    {
      "role": "user",
      "content": f'Please tell me who are your and what are some of your capabilities?'
    }
  ],
  "max_tokens": 1024,
  "temperature": 0.20,
  "top_p": 0.70,
  "stream": False
}

The payload is made up of two main parts, the messages and the hyperparameters. 

The messages key and associated list value is the set of all messages between the "user" and the "assistant". The "user" being the person interacting with the model and the "assistant" being the VLM. 

In this notebook we will only send single messages with the "user" role, but to implement a full chatting experience, the return of the VLM would be appended to these messages as the "assistant" then followed up by the next "user" message. This creates a multi turn chat that has the history of the conversation for the VLM to use. 

In the payload below the "messages" field are a set of hyperparameters that can be controlled to tune the VLM. 

- **max_tokens**: Maximum number of tokens to generate for the response. 
- **temperature**: The randomness of the output. Higher temperature allows for less likely values to be chosen in the output.   
- **top_p**: Also controls the randomness of the output. Higher top_p will make the LLM choose more likely values. 
- **stream**: Streaming responses can be used to get tokens as soon as they are generated instead of waiting for the complete response. 

For simplicity, this notebook will **NOT** use streaming responses. To see how to do this, visit the documentation pages linked in the cell above.

In [ ]:
response = requests.post(neva_api_url, headers=headers, json=payload)
response = response.json()

#print the full JSON response 
json_string = json.dumps(response, indent=4, sort_keys=True)
print(json_string)
print("\n"*2 + "#"*63)
#print only the reply 
print(response["choices"][0]["message"]["content"])

With the headers and payload setup, the Python requests library can be used to send a POST request to the VLM API url. The response will be in JSON format and can be parsed and then accessed to get the reply from the VLM. 

To summarize, Part 1 covered how to call and interact with VLM NIMs using just text input. The API schema and style of requests shown in this part is the same if you were to interact with the LLM NIMs as well. 

## Image Capabilities

The VLMs are unique because unlike LLMs, they can accept visual and text input. This section will cover the basics on providing images to the VLM.

### Image Preprocessing

Images are often provided to the multimodal models at a lower resolutions such as 224x224 or 336x336. This is based on the input size of the vision encoder used in the VLM.   

To reduce our request size, we can preprocess the image to the input resolution used by the VLM. This is not strictly necessary as the NIM itself will process the image to the correct input size but we can reduce latency and API calls by converting our image to JPEG and downsizing it before uploading it through the request. 
After the image processing is done, it is converted to a base 64 string. A base 64 string encoded image is a common way to serialize images when they are included directly with a REST API request. 

The maximum image size supported when included directly in the reponse in 180,000 bytes. For larger files, the <a href=https://docs.api.nvidia.com/cloud-functions/reference/createasset> large asset API </a> can be used. This requires a few more API calls but allows for large files to be given to the NIMs.  

For this notebook, our image sizes can be reduced sufficiently to allow for upload directly in the chat completion requests so the large asset API is not needed. 

We will use this image for our testing: 

![Test Image](imgs/testing_image.jpg)

In [ ]:
from supplementary_functions import process_image

In [ ]:
resized_image_b64 = process_image("imgs/testing_image.jpg")

After processing the image, this is what the VLM will see:

<img src="imgs/testing_image_resized.jpg">


In [ ]:
headers = {
  "Authorization": f"Bearer {api_key}",
  "Accept": "application/json"
}

The headers are configured the same way as in Part 0. 

In [ ]:
image_b64 = process_image("imgs/testing_image.jpg") #get the base 64 representation of our reduced size image 
payload = {
  "messages": [
    {
      "role": "user",
      "content": f'Describe what you see in this image. <img src="data:image/jpeg;base64,{image_b64}" />'
    }
  ],
  "max_tokens": 1024,
  "temperature": 0.20,
  "top_p": 0.70,
  "seed": 0,
  "stream": False
}

In the payload, our content field now contains our image by supplying an image tag with our prompt and referencing the base 64 string.

```<img src="data:image/jpeg;base64,{image_b64}"/```

Now when the POST request is sent, the VLM will take into account the image in our prompt!

In [ ]:
response = requests.post(neva_api_url, headers=headers, json=payload)
response = response.json()

#print only the reply 
print(response["choices"][0]["message"]["content"])

### Exercise - Try on your own image with your own queries

In this part, try these APIs on your own images. Play with the 4 hyperparameters and see what type of responses it generates

In [ ]:
image_b64 = process_image("<<ADD YOUR PATH>>") #upload an image of your choice using the upload button and put the filepath for the same.
payload = {
  "messages": [
    {
      "role": "user",
      "content": f'#FIXME <img src="data:image/jpeg;base64,{image_b64}" />'
    }
  ],
################ TEST PARAMS ################
  "max_tokens": 1024,
  "temperature": 0.20,
  "top_p": 0.70,
  "seed": 0,
################ TEST PARAMS ################
  "stream": False
}

response = requests.post(neva_api_url, headers=headers, json=payload)
response = response.json()

#print only the reply 
print(response["choices"][0]["message"]["content"])

### Abstracting the REST API

For easier use, we can wrap the API requests and image processing into a simple callable class.

In [ ]:
class VLM:
    def __init__(self, url, api_key):
        """ Provide NIM API URL and an API key"""
        self.api_key = api_key
        self.url = url 
        self.headers = {"Authorization": f"Bearer {self.api_key}", "Accept": "application/json"}

    def _encode_image(self, image):
        """ Resize image, encode as jpeg to shrink size then convert to b64 for upload """

        if isinstance(image, str): #file path
            image = Image.open(image).convert("RGB")
        elif isinstance(image, Image.Image): #pil image 
            image = image.convert("RGB")
        elif isinstance(image, np.ndarray): #cv2 / np array image 
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(image)
        else:
            print(f"Unsupported image input: {type(image)}")
            return None 
            
        image = image.resize((336,336))
        buf = io.BytesIO()
        image.save(buf, format="JPEG")
        image = buf.getvalue()
        image_b64 = base64.b64encode(image).decode()
        assert len(image_b64) < 180_000, "Image too large to upload."
        return image_b64

    def __call__(self, prompt, image):
        """ Call VLM object with the prompt and path to image """ 
        image_b64 = self._encode_image(image)

        #For simplicity, the image will be appended to the end of the prompt. 
        payload = {
              "messages": [
                {
                  "role": "user",
                  "content": f'{prompt} Here is the image: <img src="data:image/jpeg;base64,{image_b64}" />'
                }
              ],
              "max_tokens": 128,
              "temperature": 0.20,
              "top_p": 0.70,
              "stream": False
        }

        response = requests.post(self.url, headers=headers, json=payload)
        response = response.json()
        reply = response["choices"][0]["message"]["content"]
        return reply, response #return reply and the full response

Now we have an easy to use VLM class that wraps a multimodal NIM. 

Lets run some images against Neva, Kosmos and Fuyu to compare. 

#### Comparing VLMs

In [ ]:
#Create a VLM object for each supported model 
phi3 = VLM(phi3_api_url, api_key)
neva = VLM(neva_api_url, api_key)
fuyu8b = VLM(fuyu8b_api_url, api_key)

Adjust the *custom_prompt* variable if you want to try different prompts.   
Adjust the *image_path* variable if you want to try different images. 

In [ ]:
custom_prompt = "Can you tell me about the image?" #CHANGE ME
image_path = "imgs/testing_image.jpg" #CHANGE ME

In [ ]:
compare_models = []

In [ ]:
#NEVA
start_time = time()
response, _ = neva(custom_prompt, image_path)
compare_models.append({
    "MODEL":"NEVA",
    "Response":response,
    "Time Taken": time()-start_time
})

In [ ]:
#FUYU
start_time = time()
response, _ = fuyu8b(custom_prompt, image_path)
compare_models.append({
    "MODEL":"FUYU",
    "Response":response,
    "Time Taken": time()-start_time
})

In [ ]:
#Phi3
start_time = time() 
response, _= phi3(custom_prompt, image_path)
compare_models.append({
    "MODEL":"Phi3",
    "Response":response,
    "Time Taken": time()-start_time
})

Now we can see the time each model takes to send a request as well as the difference in their outputs. 

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.DataFrame(compare_models)

### Grounding with Kosmos

The Kosmos model is designed with grounding capability. Allowing it to localize areas in the image. This is particularly usefull for questions that require more precise answers such as counting and positioning. 

In the full response from Kosmos, we get bounding boxes of objects detected in addition to the text reply. This is a unique feature of the Kosmos model. The bounding box overlay can be viewed on the NIM demo for Kosmos https://build.nvidia.com/microsoft/microsoft-kosmos-2. 

#### List the Bounding Boxes

In [ ]:
kosmos2 = VLM(kosmos2_api_url, api_key)
kosmos2_response, full_response = kosmos2("Where are the robots and the man?", "imgs/testing_image.jpg")
print(json.dumps(full_response, indent=4, sort_keys=True))

The bounding boxes are associated with specific substrings in the output. Allowing parts of the output such as "robots" to then be grounded by a set of bounding boxes that indicate where robots are present in the image.

#### Visualize the Bounding Boxes

In the above response, the kosmos model predicted 2 distinct entities - "robots" and "human". Let's draw bounding boxes around each of the objects.

In [ ]:
im = Image.open('imgs/testing_image.jpg')
width, height = im.size

# Create figure and axes
fig, ax = plt.subplots()

# Display the image
ax.imshow(im)

entities = full_response["choices"][0]["message"]["entities"]
caption = full_response["choices"][0]["message"]["content"]

bbox_colors = ['b', 'r', 'g']
count=0

import matplotlib as mpl
mpl.rcParams['text.color'] = "yellow"

for objects in entities:
    for bbox in objects["bboxes"]:
        
        x0_scaled = bbox[0]*width
        y0_scaled = bbox[1]*height
        x_len = (bbox[2]*width)-x0_scaled
        y_len = (bbox[3]*height)-y0_scaled
        
        rect = patches.Rectangle((x0_scaled, y0_scaled), x_len, y_len, linewidth=1, edgecolor=bbox_colors[count], facecolor='none')
        plt.text(x0_scaled, y0_scaled, objects["phrase"])
        
        # Add the patch to the Axes
        ax.add_patch(rect)
    count+=1

plt.show()

### Utilising LLaMA-3.2 Models

Meta recently released its [Llama 3.2 ](https://developer.nvidia.com/blog/llama-3-2-full-stack-optimizations-unlock-high-performance-on-nvidia-gpus) series of vision language models (VLMs), which come in 11B parameter and 90B parameter variants. These models are multimodal, supporting both text and image inputs. They can be quickly accessed via the NIM endpoints or can be downloaded to run on your own setups.

#### Cloud Endpoints

In [ ]:
import requests, base64

invoke_url = "https://ai.api.nvidia.com/v1/gr/meta/llama-3.2-90b-vision-instruct/chat/completions"
stream = False

with open("imgs/testing_image.jpg", "rb") as f:
  image_b64 = base64.b64encode(f.read()).decode()

assert len(image_b64) < 180_000, \
  "To upload larger images, use the assets API (see docs)"
  

headers = {
  "Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}",
  "Accept": "text/event-stream" if stream else "application/json"
}

payload = {
  "model": 'meta/llama-3.2-90b-vision-instruct',
  "messages": [
    {
      "role": "user",
      "content": f'What is in this image? <img src="data:image/jpg;base64,{image_b64}" />'
    }
  ],
  "max_tokens": 512,
  "temperature": 1.00,
  "top_p": 1.00,
  "stream": stream
}

response = requests.post(invoke_url, headers=headers, json=payload)

if stream:
    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))
else:
    print(response.json())


#### Local Containers

In [ ]:
import random
import socket
import os 

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

In [ ]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

In [ ]:
!mkdir -p "$LOCAL_NIM_CACHE"
!chmod 777 "$LOCAL_NIM_CACHE"

In [ ]:
! docker run -it --rm -d \
--gpus 1 \
--name=vlm_nim \
--shm-size=16GB  \
-e NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/meta/llama-3.2-11b-vision-instruct:1.1.1

# -e NIM_MODEL_PROFILE=tensorrt_llm-a100-bf16-tp1-throughput \
# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

In [ ]:
! docker ps -a | awk 'NR==1 || /llama/'

In [ ]:
from openai import OpenAI
import base64

# Set your API key and base URL
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:{}/v1".format(os.environ['CONTAINER_PORT'])

# Init a client
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

base64_image = None
with open("imgs/testing_image.jpg", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
# Request the vila model
response = client.chat.completions.create(
    model="meta/llama-3.2-11b-vision-instruct",  # Specify the model hosted
    messages=[
        {
            "role": "user", 
            "content": [
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
                },
                {
                    "type": "text",
                    "text": "Describe the content of this image."
                }
            ]
        }
    ],
    temperature=0.4,
    max_tokens=256
)

# Print the response
print("Response:\n\n", response.choices[0].message.content)

## [Optional] VLM for Videos

Now that the basics are covered, let’s explore a more practical use of VLMs—extracting insights from live video streams. Streaming sources like security cameras and drones produce vast amounts of video data. Manually reviewing this footage is impractical, making it difficult to extract meaningful insights.

A common use case can be real-time detection of specific events, such as traffic incidents or accidents. Instead of continuous human monitoring, we can deploy a VLM-based NIM to watch the stream and respond when specific events—like an accident—are detected. Simply provide the prompt, and the VLM will handle the rest.

### Simple Video Pipeline 

We can start by building a pipeline that can open a video file and call the VLM. 

In [ ]:
#! pip install opencv-python

In [ ]:
import os
import base64
import cv2
from openai import OpenAI

class LLAMA_VLM:
    def __init__(self, container_port="1234", api_key="EMPTY"):
        self.client = OpenAI(
            api_key=api_key,
            base_url=f"http://localhost:{container_port}/v1"
        )

    def __call__(self, prompt, frame):
        _, buffer = cv2.imencode('.jpg', frame)
        base64_image = base64.b64encode(buffer).decode('utf-8')

        response = self.client.chat.completions.create(
            model="meta/llama-3.2-11b-vision-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ],
            temperature=0.4,
            max_tokens=256
        )
        return response.choices[0].message.content

In [ ]:
from IPython.display import Video

video_path = "imgs/traffic.mp4"
Video(video_path, embed=True, width=400)

In [ ]:

video_path = "imgs/traffic.mp4"
prompt = "Is there an accident in the image? Answer yes or no."

vlm = LLAMA_VLM(container_port=os.environ["CONTAINER_PORT"])

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frames_to_skip = int(fps * 4)

count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if count % frames_to_skip == 0:
        reply = vlm(prompt, frame)
        timestamp = count / fps
        print(f"Frame {count} | Time {timestamp:.2f}s: {reply}")

    count += 1

cap.release()

# Further Readings

Please refer to the following sources for more detailed understanding about Vision Language Models (VLMs):
* [An Introduction to Vision-Language Modeling](https://arxiv.org/abs/2405.17247)

---
## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.